# Advanced 06 — Conditional Expectation as Projection and Martingales

Practice notebook for the **"Conditional Expectation as Projection and Martingales"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


## Part 1 — Conditional expectation as an $L^2$ projection

The PDF defines the **conditional expectation** $Y = E[X \mid \mathcal{G}]$ for an
integrable $X$ and a sub-sigma-algebra $\mathcal{G} \subseteq \mathcal{F}$ by two properties:

1. $Y$ is $\mathcal{G}$-measurable.
2. For every $G \in \mathcal{G}$, $\displaystyle\int_G Y\,dP = \int_G X\,dP$.

Equivalently, $E[X \mid \mathcal{G}]$ is the **orthogonal projection** of $X$ onto the
closed subspace of $\mathcal{G}$-measurable, square-integrable random variables: among all
$\mathcal{G}$-measurable $Z$, it minimizes $E[(X-Z)^2]$. In particular, taking
$\mathcal{G} = \sigma(Y)$ means the conditional expectation is a function of $Y$ alone.

A key consequence is the **tower property**: if $\mathcal{H} \subseteq \mathcal{G}$ then

$$
E\big[E[X \mid \mathcal{G}] \mid \mathcal{H}\big] = E[X \mid \mathcal{H}],
\qquad\text{and in particular}\qquad E\big[E[X \mid \mathcal{G}]\big] = E[X].
$$

Below we generate $(X, Y)$ with a nonlinear dependence, estimate
$E[X \mid \sigma(Y)]$ by binning $Y$ (a nonparametric estimate of the projection), and
verify the tower property $E[E[X \mid \sigma(Y)]] = E[X]$.


In [ ]:
# Joint model: Y ~ Uniform(0,1); X = sin(2*pi*Y) + noise
rng = np.random.default_rng(7)
N = 200_000
Y = rng.uniform(0, 1, size=N)
eps = rng.normal(0.0, 0.25, size=N)
X = np.sin(2 * np.pi * Y) + eps

# Estimate E[X | sigma(Y)] by binning Y into K bins (a function of Y alone -> G-measurable)
K = 40
edges = np.linspace(0, 1, K + 1)
bins = np.clip(np.digitize(Y, edges) - 1, 0, K - 1)
cond_est = np.array([X[bins == k].mean() if np.any(bins == k) else 0.0 for k in range(K)])

# G-measurable predictor: map each observation to its bin's conditional mean
proj = cond_est[bins]

# L2 projection check: MSE of E[X|G] <= MSE of any other G-measurable predictor.
# Compare against the trivial G-measurable predictor g(Y) = E[X] (constant in Y).
mse_proj = np.mean((X - proj) ** 2)
mse_const = np.mean((X - X.mean()) ** 2)
print("MSE of E[X|sigma(Y)] (projection) :", round(mse_proj, 5))
print("MSE of g(Y) = E[X] (constant)     :", round(mse_const, 5))
print("projection has lower MSE          :", bool(mse_proj < mse_const))

# Tower property: E[ E[X|G] ] == E[X]
print()
print("E[X]                (sample) :", round(X.mean(), 5))
print("E[ E[X|sigma(Y)] ]  (sample) :", round(proj.mean(), 5))
print("abs diff                    :", round(abs(X.mean() - proj.mean()), 6))

# Visual: the projection is the conditional-mean function of Y
y_mid = 0.5 * (edges[:-1] + edges[1:])
plt.scatter(Y[:3000], X[:3000], s=6, alpha=0.25, label="sample (X, Y)")
plt.stairs(cond_est, edges, color="crimson", lw=2,
           label=r"$\hat E[X\mid\sigma(Y)]$ (binning)")
plt.xlabel("Y"); plt.ylabel("X")
plt.title(r"$E[X\mid\sigma(Y)]$ is the $L^2$ projection onto $\sigma(Y)$-measurable fns")
plt.legend()
plt.tight_layout(); plt.show()


## Part 2 — Symmetric random walk is a martingale

A **filtration** $(\mathcal{F}_n)$ is an increasing sequence of sigma-algebras; think of
$\mathcal{F}_n$ as "information available up to time $n$". A sequence $(M_n)$ is a
**martingale** w.r.t. $(\mathcal{F}_n)$ if:

1. $M_n$ is $\mathcal{F}_n$-measurable,
2. $E[|M_n|] < \infty$,
3. $E[M_{n+1} \mid \mathcal{F}_n] = M_n$ almost surely.

The archetypal example is the **symmetric random walk**
$S_n = \sum_{i=1}^{n} \xi_i$ with $\xi_i \in \{-1,+1\}$ i.i.d. and equiprobable, w.r.t.
the natural filtration $\mathcal{F}_n = \sigma(\xi_1, \dots, \xi_n)$. It models a fair game:
the conditional expectation of tomorrow's position given today's information is today's
position. We verify two facts numerically:

- the **unconditional mean is constant**: $E[S_n] = 0$ for every $n$;
- the **martingale property**: $E[S_{n+1} \mid \mathcal{F}_n] = S_n$, i.e. averaging
  $S_{n+1}$ *within each realized path's current state* returns $S_n$.


In [ ]:
rng = np.random.default_rng(11)
N_paths, n_steps = 50_000, 20
xi = rng.choice([-1, 1], size=(N_paths, n_steps))
S = np.cumsum(xi, axis=1)              # S_n for n = 1..n_steps
S0 = np.zeros(N_paths)
S_all = np.column_stack([S0, S])       # include S_0 = 0

# Unconditional mean E[S_n] for each n
means = S_all.mean(axis=0)
print("E[S_n] for n=0..20:")
print(np.round(means, 4))
print("max |E[S_n]| over all n  :", round(np.abs(means).max(), 4))

# Martingale property E[S_{n+1} | F_n] = S_n:
# Group paths by their value of S_n (the F_n-measurable state) and average S_{n+1}.
n_check = 10
for n in range(n_check):
    states = S_all[:, n]
    next_vals = S_all[:, n + 1]
    unique = np.unique(states)
    cond_mean = np.array([next_vals[states == s].mean() for s in unique])
    # Conditional mean should equal the state itself (up to MC noise)
    max_err = np.abs(cond_mean - unique).max()
    print(f"n={n:2d}: max|E[S_{{n+1}}|S_n=s] - s| = {max_err:.4f}")

# Plot a handful of sample paths
plt.plot(S_all[:50].T, alpha=0.5, lw=0.8)
plt.axhline(0, color="k", lw=1)
plt.xlabel("n"); plt.ylabel(r"$S_n$")
plt.title("Symmetric random walk: a martingale with constant mean 0")
plt.tight_layout(); plt.show()


## Part 3 — Doob's maximal inequality

The PDF states a simplified **Doob $L^p$ inequality**: for a martingale $(M_n)$ with
$M_0 = 0$ and $1 < p < \infty$,

$$
E\big[\max_{1\leq k\leq n} |M_k|^p\big] \;\leq\; C_p\, E[|M_n|^p].
$$

For $p = 2$ this comes with an explicit, sharper **maximal** bound on the running
maximum $M_n^* = \max_{k\leq n} |M_k|$:

$$
P\big(\max_{k\leq n} |M_k| \geq a\big) \;\leq\; \frac{E[M_n^2]}{a^2}.
$$

Below we use the symmetric random walk ($M_n = S_n$) to check this: we estimate the
left-hand side by simulation and compare to the theoretical bound
$E[S_n^2]/a^2 = n/a^2$ (since $\operatorname{Var}(S_n) = n$). The bound should always
hold; the simulation should sit *below* it.


In [ ]:
rng = np.random.default_rng(23)
N_paths, n = 100_000, 50
xi = rng.choice([-1, 1], size=(N_paths, n))
S = np.cumsum(xi, axis=1)                       # M_k for k = 1..n
M = np.column_stack([np.zeros(N_paths), S])     # include M_0 = 0
running_max = np.abs(M[:, 1:n + 1]).cummax(axis=1)  # max_{k<=j} |M_k| per path

# Theoretical bound E[M_n^2]/a^2 = Var(S_n)/a^2 = n / a^2
a_grid = np.arange(3, 16)
empirical = np.array([(running_max[:, n - 1] >= a).mean() for a in a_grid])
bound = n / a_grid ** 2

print(" a | P(max|M_k|>=a) [sim] | n/a^2 [bound] | holds?")
for a, emp, bnd in zip(a_grid, empirical, bound):
    print(f"{a:2d} |     {emp:.4f}        |     {bnd:.4f}      | {emp <= bnd + 0.005}")

plt.plot(a_grid, empirical, "o-", label=r"$P(\max_{k\leq n}|M_k|\geq a)$ (sim)")
plt.plot(a_grid, bound, "s--", color="crimson",
         label=r"Doob bound $E[M_n^2]/a^2 = n/a^2$")
plt.xlabel("threshold $a$"); plt.ylabel("probability")
plt.title(f"Doob maximal inequality for the symmetric walk (n={n})")
plt.legend()
plt.tight_layout(); plt.show()

# Also check the L^2 form E[max|M_k|^2] <= C_2 * E[M_n^2]; Doob gives C_2 = 4 for p=2
E_max_sq = np.mean(running_max[:, n - 1] ** 2)
E_Mn_sq = np.mean(S[:, n - 1] ** 2)
print()
print("E[max|M_k|^2] (sim)     :", round(E_max_sq, 3))
print("E[M_n^2]    (sim)       :", round(E_Mn_sq, 3))
print("ratio E[max^2]/E[M_n^2] :", round(E_max_sq / E_Mn_sq, 3), " (Doob C_2 = 4)")


## Part 4 — Optional stopping: you cannot beat a fair game

A random time $\tau$ is a **stopping time** w.r.t. $(\mathcal{F}_n)$ if
$\{\tau \leq n\} \in \mathcal{F}_n$ for all $n$ — i.e. the decision to stop by time $n$
depends only on information available up to $n$. The PDF's simplified **optional
stopping theorem** states that if $(M_n)$ is a martingale and $\tau$ is a **bounded**
stopping time ($\tau \leq N$ a.s.), then

$$
E[M_\tau] = E[M_0].
$$

We test this with the symmetric walk stopped at the **first hitting time** of a level
$\pm b$:

$$
\tau = \inf\{\, k \geq 0 : |S_k| \geq b\,\},
$$

truncated at a finite horizon $N$ so that $\tau$ is bounded. If the walk has not hit
$\pm b$ by step $N$, we stop at $N$ anyway. The theorem predicts
$E[S_\tau] = E[S_0] = 0$.


In [ ]:
rng = np.random.default_rng(29)
N_paths = 200_000
N = 200           # bounded horizon -> tau <= N a.s.
b = 5             # stopping barrier |S_k| >= b

xi = rng.choice([-1, 1], size=(N_paths, N))
S = np.cumsum(xi, axis=1)
S = np.column_stack([np.zeros(N_paths), S])   # S_0=0, ..., S_N

# tau = first k where |S_k| >= b, else N (bounded stopping time)
abs_S = np.abs(S)
hit = abs_S >= b
# has_hit: whether the path ever hits the barrier within 0..N
has_hit = hit.any(axis=1)
first_hit = np.argmax(hit, axis=1)            # first index where hit (argmax returns 0 if no hit)
tau = np.where(has_hit, first_hit, N)

# M_tau = S at the stopping time
rows = np.arange(N_paths)
M_tau = S[rows, tau]

print("E[S_0]                    :", round(S[:, 0].mean(), 5))
print("E[S_tau] (bounded stop)   :", round(M_tau.mean(), 5))
print("abs diff                  :", round(abs(M_tau.mean()), 5))
print("P(tau < N) (hit barrier)  :", round((tau < N).mean(), 4))
print("P(tau = N) (truncated)    :", round((tau == N).mean(), 4))

# Distribution of tau (truncated)
vals, counts = np.unique(tau, return_counts=True)
print()
print("tau distribution (first few and last few):")
for v, c in list(zip(vals, counts))[:5] + [("...", "...")] + list(zip(vals, counts))[-3:]:
    print(f"  tau={v:>3}: freq={c/N_paths:.4f}")

# Histogram of stopped values
plt.hist(M_tau, bins=41, density=True, alpha=0.7, color="steelblue")
plt.axvline(0, color="crimson", linestyle="--", label=r"$E[S_0]=0$")
plt.axvline(M_tau.mean(), color="k", linestyle=":", label=fr"$\hat E[S_\tau]={M_tau.mean():.3f}$")
plt.xlabel(r"$S_\tau$"); plt.ylabel("density")
plt.title(r"Optional stopping: $E[S_\tau]=E[S_0]=0$ for bounded $\tau$")
plt.legend()
plt.tight_layout(); plt.show()


## Part 5 — Martingale convergence

The PDF's **martingale convergence theorem**: if $(M_n)$ is bounded in $L^1$,
$\sup_n E[|M_n|] < \infty$, then there exists an integrable $M_\infty$ with

$$
M_n \to} M_\infty
\quad\text{and}\quad
M_n \to M_\infty.
$$

A canonical $L^1$-bounded (in fact $L^2$-bounded) martingale is the
**Radon-Nikodym density / likelihood-ratio martingale**: fix a target Bernoulli($p$)
and a prior $\pi_n$ = posterior mean after $n$ Bernoulli($1/2$) "coin-flip" observations
under a Beta prior. A cleaner and standard construction is the
**Polya's-urn / Beta-Bernoulli posterior mean**, which is a bounded martingale
converging to the random limiting frequency. Here we use the simplest $L^2$-bounded
martingale: the **posterior mean of a Bernoulli parameter under a Beta($\alpha,\beta$)
prior**, with $n$ Bernoulli($p$) observations. It is bounded in $[0,1]$, hence
$L^1$-bounded, and converges a.s. to $p$.

**Your turn:** change the true $p$, the prior $\alpha, \beta$, and the number of
paths. Does $M_n$ still converge to $p$? What happens to the spread of $M_n$ across
paths as $n$ grows? Try $\alpha=\beta=1$ (uniform prior) vs a strongly informative
prior $\alpha=\beta=50$.


In [ ]:
# Posterior mean of a Bernoulli(p) under a Beta(a,b) prior is a [0,1]-bounded martingale
rng = np.random.default_rng(101)
p_true = 0.35
a_prior, b_prior = 1.0, 1.0   # Uniform prior on [0,1]

N_paths, n_obs = 5_000, 500
draws = rng.binomial(1, p_true, size=(N_paths, n_obs)).astype(float)
# cumulative successes S_n = sum_{i<=n} X_i
successes = np.cumsum(draws, axis=1)
n = np.arange(1, n_obs + 1)
# Posterior mean M_n = (a + S_n) / (a + b + n)
M = (a_prior + successes) / (a_prior + b_prior + n)
# Prior mean M_0
M0 = np.full(N_paths, a_prior / (a_prior + b_prior))
M_all = np.column_stack([M0, M])

print("M_0  (prior mean)        :", round(M0.mean(), 4))
print("E[|M_n|] at n=0,100,500  :",
      [round(np.abs(M_all[:, k]).mean(), 4) for k in [0, 100, 500]])
print("sup_n E[|M_n|] (<=1)     :", round(np.abs(M_all).mean(axis=0).max(), 4))

# Almost-sure convergence: each path should approach p_true
final_err = M_all[:, -1] - p_true
print()
print(f"M_{{n={n_obs}}} mean             :", round(M_all[:, -1].mean(), 4),
      f"(true p = {p_true})")
print(f"mean |M_{{n={n_obs}}} - p|          :", round(np.abs(final_err).mean(), 5))
print(f"P(|M_{{n={n_obs}}} - p| < 0.02)     :", round((np.abs(final_err) < 0.02).mean(), 4))

# L1 convergence: E|M_n - p| -> 0
horizons = [10, 50, 200, 500]
print()
print("L1 convergence E|M_n - p|:")
for k in horizons:
    print(f"  n={k:3d}: E|M_n - p| = {np.abs(M_all[:, k] - p_true).mean():.5f}")

# Plot sample paths converging to p_true
plt.plot(M_all[:30].T, alpha=0.5, lw=0.7)
plt.axhline(p_true, color="k", lw=1.2, label=fr"true $p={p_true}$")
plt.xlabel("n"); plt.ylabel(r"$M_n$ (posterior mean)")
plt.title("Bounded martingale converges a.s. to $p$")
plt.legend()
plt.tight_layout(); plt.show()


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Let $\mathcal{G}=\sigma(Y)$ with $Y\sim\text{Uniform}(0,1)$ and $X=\cos(\pi Y)+\varepsilon$, $\varepsilon\sim\mathcal{N}(0,\sigma^2)$ independent of $Y$. State $E[X\mid\mathcal{G}]$ as a function of $Y$ and verify the tower property $E[E[X\mid\mathcal{G}]]=E[X]$ analytically.

2. For the symmetric random walk $S_n=\sum_{i=1}^n\xi_i$ with $\xi_i\in\{-1,+1\}$ i.i.d., prove $E[S_{n+1}\mid\mathcal{F}_n]=S_n$ using only the fact that $E[\xi_{n+1}\mid\mathcal{F}_n]=E[\xi_{n+1}]=0$ and that $S_n$ is $\mathcal{F}_n$-measurable.

3. State Doob's maximal $L^2$ inequality $P(\max_{k\leq n}|M_k|\geq a)\leq E[M_n^2]/a^2$ and verify the bound for the symmetric walk with $n=50$, $a=8$ by computing both sides (theory says $n/a^2$).

4. Define a bounded stopping time $\tau=\inf\{k:|S_k|\geq 3\}\wedge N$ with $N=100$. Why is $\tau$ a stopping time? State the optional stopping theorem and explain in one sentence why it implies $E[S_\tau]=0$.

5. Give an example of an $L^1$-bounded martingale that is *not* $L^2$-bounded, and explain why the martingale convergence theorem still applies but Doob's $L^2$ maximal inequality does not.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Since $\varepsilon$ is independent of $Y$, $E[\varepsilon\mid Y]=E[\varepsilon]=0$, so

$$E[X\mid\mathcal{G}] = E[\cos(\pi Y)\mid Y] + E[\varepsilon\mid Y] = \cos(\pi Y).$$

Tower: $E[E[X\mid\mathcal{G}]] = E[\cos(\pi Y)] = \int_0^1 \cos(\pi y)\,dy = \tfrac{1}{\pi}[\sin(\pi y)]_0^1 = 0$, and $E[X] = E[\cos(\pi Y)] + E[\varepsilon] = 0 + 0 = 0$. Both sides equal $0$. ✓

**2.** Write $S_{n+1}=S_n+\xi_{n+1}$. Since $S_n$ is $\mathcal{F}_n$-measurable and $\xi_{n+1}$ is independent of $\mathcal{F}_n$ with mean $0$,

$$E[S_{n+1}\mid\mathcal{F}_n]=S_n+E[\xi_{n+1}\mid\mathcal{F}_n]=S_n+E[\xi_{n+1}]=S_n+0=S_n.$$

This is precisely the martingale property.

**3.** Doob: $P(\max_{k\leq n}|M_k|\geq a)\leq E[M_n^2]/a^2$. For the symmetric walk $M_k=S_k$, $E[S_n^2]=\operatorname{Var}(S_n)=n$, so the bound is $n/a^2$. With $n=50$, $a=8$: bound $=50/64\approx 0.781$. A Monte Carlo with $N$ paths estimates the left side as the fraction of paths whose running maximum reaches $8$; this empirical value sits at or below $0.781$ (typically substantially below, since the bound is not tight).

**4.** $\tau$ is a stopping time because $\{\tau\leq n\}$ depends only on $(S_0,\dots,S_n)$ — specifically whether $|S_k|\geq 3$ for some $k\leq n$, or $n\geq N$. The optional stopping theorem for a *bounded* stopping time gives $E[M_\tau]=E[M_0]$. Here $M_0=S_0=0$, so $E[S_\tau]=0$: a fair game cannot be beaten by a bounded stopping rule based on current information.

**5.** The likelihood-ratio martingale $M_n = \prod_{i=1}^n \frac{f_1(X_i)}{f_0(X_i)}$ of i.i.d. observations under two hypotheses is a nonnegative martingale with $E[M_n]=1$ for all $n$, hence $L^1$-bounded, so it converges a.s. by the martingale convergence theorem. But $E[M_n^2]$ typically blows up (multiplicative products have heavy tails), so it is not $L^2$-bounded and Doob's $L^2$ maximal inequality cannot be applied to control its running maximum.

</details>
